# **MODELO TFT PREDITIVO**
*by Miguel Ferreira*

Antes de começarmos a criar nosso modelo, devemos seguir este pequeno passo-a-passo para garantir o **isolamento do ambiente de desenvolvimento através da criação de um ambiente virtual e kernel próprios do projeto**. Segue-se o procedimento.

## Setup de Ambiente Python Isolado (venv + Jupyter)

Este guia descreve, de forma direta e reprodutível, como configurar um ambiente Python isolado para execução de notebooks com kernel próprio.

---

### 1. Verificação inicial do Python

No terminal, verificar instalações disponíveis:

```powershell
where.exe python
python --version
py --version
```

---

### 2. Criação do ambiente virtual

A partir da raiz do projeto:

```powershell
cd C:\projects\Libellula
py -3.11 -m venv .venv
```

---

### 3. Ativação do ambiente

```powershell
.\.venv\Scripts\activate
```

Confirmação:

```
(.venv)
```

---

### 4. Validação do ambiente isolado

```powershell
python --version
where.exe python
```

Validação adicional:

```powershell
python -c "import sys; print(sys.executable)"
```

O caminho retornado deve apontar para o diretório `.venv`.

---

### 5. Instalação do kernel do Jupyter

Com o ambiente ativo:

```powershell
pip install ipykernel
python -m ipykernel install --user --name tft_env --display-name "Python (tft_env)"
```

---

### 6. Inicialização do Jupyter

```powershell
jupyter lab
```

---

### 7. Seleção do kernel

No notebook, selecionar:

```
Python (tft_env)
```

---

### 8. Verificação dentro do notebook

```python
import sys
print(sys.executable)
```

O caminho deve corresponder ao ambiente `.venv`.

---

### Resultado

Ambiente Python isolado, com kernel próprio, pronto para execução de notebooks e desenvolvimento reprodutível.

Cumpridas todas as etapas acima, seguimos com o teste final básico:

In [1]:
import sys
print(sys.executable)

C:\projects\Libellula\.venv\Scripts\python.exe


Tudo ok. Estamos prontos para começar.

## Preparativos

Começamos pela importação das bibliotecas, mas, antes, precisamos estabelecer como as instalações das bibliotecas e dependências deve ser feita de forma mais eficiente. **Atenção: tudo isto deve ser feito no Powershell (ou em qualquer outro terminal de sua preferência).**

1. Ativamos o ambiente virtual:
   
   ``` powershell
   .\.venv\Scripts\activate
   ```
2. Depois, fazemos as instalações:

   
   ``` powershell
   pip install pandas numpy torch pytorch-lightning pytorch-forecasting matplotlib pyarrow
   ```  
3. Por fim, testamos se e quais instalações foram feitas:

   
   ``` powershell
   pip list
   ```   
   Para este último comando, tivemos o seguinte output:
```
Package                 Version
----------------------- -----------
aiohappyeyeballs        2.6.1
aiohttp                 3.13.5
aiosignal               1.4.0
asttokens               3.0.1
attrs                   26.1.0
colorama                0.4.6
comm                    0.2.3
contourpy               1.3.3
cycler                  0.12.1
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
filelock                3.29.0
fonttools               4.62.1
frozenlist              1.8.0
fsspec                  2026.4.0
idna                    3.13
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
joblib                  1.5.3
jupyter_client          8.8.0
jupyter_core            5.9.1
kiwisolver              1.5.0
lightning               2.6.1
lightning-utilities     0.15.3
MarkupSafe              3.0.3
matplotlib              3.10.9
matplotlib-inline       0.2.1
mpmath                  1.3.0
multidict               6.7.1
nest-asyncio            1.6.0
networkx                3.6.1
numpy                   2.4.4
packaging               26.2
pandas                  3.0.2
parso                   0.8.6
pillow                  12.2.0
pip                     24.0
platformdirs            4.9.6
prompt_toolkit          3.0.52
propcache               0.4.1
psutil                  7.2.2
pure_eval               0.2.3
pyarrow                 24.0.0
Pygments                2.20.0
pyparsing               3.3.2
python-dateutil         2.9.0.post0
pytorch-forecasting     1.7.0
pytorch-lightning       2.6.1
PyYAML                  6.0.3
pyzmq                   27.1.0
scikit-base             0.13.2
scikit-learn            1.8.0
scipy                   1.17.1
setuptools              65.5.0
six                     1.17.0
stack-data              0.6.3
sympy                   1.14.0
threadpoolctl           3.6.0
torch                   2.11.0
torchmetrics            1.9.0
tornado                 6.5.5
tqdm                    4.67.3
traitlets               5.14.3
typing_extensions       4.15.0
tzdata                  2026.2
wcwidth                 0.6.0
yarl                    1.23.0
```

Agora, finalmente, iniciamos os códigos do projeto.   

## **1. Importações**

In [2]:
import pandas as pd
import numpy as np
import torch

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

import lightning.pytorch as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

import matplotlib.pyplot as plt

C:\projects\Libellula\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## **2. Loading de dataset original**

In [3]:
# Carregar dataset base (OHLC + indicadores)
df = pd.read_csv("C:/projects/Libellula/data/processed/financial_tools_datset.csv")

# Ordenar temporalmente (CRÍTICO)
df = df.iloc[::-1].reset_index(drop=True)

# Converter data
df["Date"] = pd.to_datetime(df["Date"])

# Criar índice temporal numérico (exigência do TFT)
df["time_idx"] = np.arange(len(df))

# Criar target → retorno futuro (melhor que preço)
df["target"] = df["Price"].pct_change().shift(-1)

# Remover NaNs
df = df.dropna().reset_index(drop=True)
    
# ID da série (necessário para TFT)
df["series"] = "asset_1"

df.head()

,Date,Price,Open,High,Low,Change %,short_mavg,long_mavg,signal,EMA10,...,RSI200,%K10,%D10,%K30,%D30,%K200,%D200,time_idx,target,series
0,2020-02-11,1.0914,1.0911,1.0926,1.0890,0.05%,1.08362,1.094205,0.0,1.087593,...,47.144045,91.946309,67.586943,32.322054,27.771295,20.130814,17.296512,0,-0.003940,asset_1
1,2020-02-12,1.0871,1.0916,1.0926,1.0865,-0.39%,1.08327,1.094078,0.0,1.086747,...,46.880856,63.087248,44.652875,27.304551,24.504084,17.005814,15.261628,1,-0.002852,asset_1
2,2020-02-13,1.0840,1.0874,1.0890,1.0833,-0.29%,1.08335,1.093950,0.0,1.086668,...,46.690443,47.727273,30.482998,23.687281,23.064955,14.752907,14.365310,2,-0.000923,asset_1
3,2020-02-14,1.0830,1.0841,1.0862,1.0827,-0.09%,1.08493,1.093937,0.0,1.087261,...,46.629037,23.144105,15.712119,22.520420,21.159082,14.026163,13.178295,3,0.000369,asset_1
4,2020-02-17,1.0834,1.0832,1.0852,1.0821,0.04%,1.08688,1.093953,0.0,1.088208,...,46.650424,20.577617,10.056914,22.987165,20.147802,14.316860,12.548450,4,-0.003969,asset_1


## **3. Configuração temporal**

In [4]:
max_encoder_length = 64
max_prediction_length = 1

## 4. **Criação do dataset do TFT**

```python
cols_obj = df.select_dtypes(include=["object"]).columns
```
**O código seguinte é de suma importância para o treinamento do TFT preditivo,** pois ele trabalha apenas com valores numéricos, então temos que nos livrar de eventuais strings dentro do dataset.
1) Selecionamos todas as colunas com tipo "object" (strings), que não são compatíveis com o modelo.
```python
for col in cols_obj:
    df[col] = (
        df[col]
        .astype(str)
```
2) Garante que todos os valores são tratados como string para permitir operações de limpeza.
```python
        .str.replace("%", "", regex=False)
```
3) Remove símbolos não numéricos (ex: %), que impedem conversão para float.
```python
        .str.replace(",", ".", regex=False)
```
4) Padroniza separadores decimais (vírgula → ponto), garantindo formato numérico válido.
```python
        .str.extract(r"([-+]?\d*\.?\d+)")[0]
```
5) Extrai apenas a parte numérica da string usando regex (remove qualquer outro caractere residual).
```python
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")
```
6) Converte os valores para numérico (float). Valores inválidos viram NaN.
```python
    df = df.dropna().reset_index(drop=True)
```
7) Remove linhas com NaN gerados na conversão e reorganiza o índice.

#### Resultado:
Dataset totalmente numérico, compatível com normalização e treinamento do modelo.

#### Abaixo, aplicamos o código acima explicado

In [5]:
cols_obj = df.select_dtypes(include=["object"]).columns

for col in cols_obj:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.extract(r"([-+]?\d*\.?\d+)")[0]  # <-- pega só número
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().reset_index(drop=True)

C:\Users\Miguel\AppData\Local\Temp\ipykernel_13856\613697189.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_obj = df.select_dtypes(include=["object"]).columns


In [6]:
df.select_dtypes(include=["object"])

""
0
1
2
3
4
...
1362
1363
1364
1365


In [7]:
# Definir colunas que o modelo usa
training = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="target",
    group_ids=["series"],

    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,

    time_varying_known_reals=["time_idx"],

    time_varying_unknown_reals=[
        col for col in df.columns
        if col not in ["Date", "series", "target"]
    ]
)

## **5. Dataloader**

In [8]:
train_dataloader = training.to_dataloader(
    train=True,
    batch_size=64,
    num_workers=0
)

## 6. **Criar modelo TFT**

In [9]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.001,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(),
    log_interval=10,
    reduce_on_plateau_patience=4,
)

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


## 7. **Callbacks**

In [10]:
early_stop = EarlyStopping(
    monitor="train_loss",
    patience=5,
    mode="min"
)

lr_logger = LearningRateMonitor()

## **8. Trainer**

In [11]:
trainer = pl.Trainer(
    max_epochs=30,
    gradient_clip_val=0.1,
    callbacks=[early_stop, lr_logger]
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## 9. **Treinamento**

In [12]:
trainer.fit(
    tft,
    train_dataloaders=train_dataloader
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.

   | Name                               | Type                            | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train | 0    
1  | logging_metrics                    | ModuleList                      | 0      | train | 0    
2  | input_embeddings                   | MultiEmbedding                  | 0      | train | 0    
3  | prescalers                         | ModuleDict                      | 800    | train | 0    
4  | stati

Epoch 1:   0%|      | 0/20 [00:00<?, ?it/s, v_num=2, train_loss_step=0.00181, train_loss_epoch=0.00235]

C:\projects\Libellula\.venv\Lib\site-packages\lightning\pytorch\loops\training_epoch_loop.py:500: ReduceLROnPlateau conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.


Epoch 29: 100%|█| 20/20 [00:13<00:00,  1.53it/s, v_num=2, train_loss_step=0.0016, train_loss_epoch=0.00

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|█| 20/20 [00:13<00:00,  1.51it/s, v_num=2, train_loss_step=0.0016, train_loss_epoch=0.00


## 10. **Gerar previsões**

In [14]:
raw_predictions = tft.predict(
    train_dataloader,
    mode="raw"
)

pred = raw_predictions["prediction"].numpy()
mean_pred = pred.mean(axis=2)

## 11. **Criar dataset de previsões**

## 12. **Carregar dataset de risco**

## 13. **Alinhamento (CRÍTICO!)**

## 14. **Merge final**

## 15. **Salvar dataset final**

## 16. **Salvar modelo**